# 02 — Feature engineering, targets and chronological split

This notebook creates the paper's 7- and 15-feature sets per pair, plus a stationary 15-feature variant, and next-observation targets. It saves each cleaned pair dataset so the modelling notebooks can be run independently.

Run notebook 01 first to understand the source data. This notebook reloads the raw CSV and repeats the deterministic preparation steps.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fx_utils import (
    load_forex_data, build_pair_rates, build_features_and_targets,
    chronological_split, get_feature_sets, PAIRS, PROCESSED_DIR
)

forex_df = load_forex_data(PROJECT_ROOT / "data" / "daily_forex_rates.csv")
pair_df = build_pair_rates(forex_df)
features_df, targets_df, model_datasets = build_features_and_targets(pair_df)

print("Pair rates:", pair_df.shape)
print("All feature columns:", features_df.shape)
print("Target columns:", targets_df.shape)

Pair rates: (6057, 3)
All feature columns: (6057, 69)
Target columns: (6057, 6)


In [2]:
print("Feature-set sizes per pair:")
for pair in PAIRS:
    sizes = {name: len(cols) for name, cols in get_feature_sets(pair).items()}
    print(f"{pair}: {sizes}")

print("\nFirst date with complete feature and target rows:")
for pair in PAIRS:
    data = model_datasets[pair]
    print(pair, data.shape, data.index.min(), "to", data.index.max(),
          "| missing values:", int(data.isna().sum().sum()))

Feature-set sizes per pair:
GBP_USD: {'7': 7, '15-paper': 15, '15-stationary': 15}
USD_EUR: {'7': 7, '15-paper': 15, '15-stationary': 15}
USD_JPY: {'7': 7, '15-paper': 15, '15-stationary': 15}

First date with complete feature and target rows:
GBP_USD (5837, 25) 2005-07-29 00:00:00 to 2026-10-10 00:00:00 | missing values: 0
USD_EUR (5837, 25) 2005-07-29 00:00:00 to 2026-10-10 00:00:00 | missing values: 0
USD_JPY (5837, 25) 2005-07-29 00:00:00 to 2026-10-10 00:00:00 | missing values: 0


In [3]:
# Inspect the three feature sets.
pair = "GBP_USD"
feature_sets = get_feature_sets(pair)
features_7 = feature_sets["7"]
print("7-feature set:", features_7)
print("\n15-paper adds:", [c for c in feature_sets["15-paper"] if c not in features_7])
print("\n15-stationary adds:", [c for c in feature_sets["15-stationary"] if c not in features_7])
display(model_datasets[pair].head())

7-feature set: ['GBP_USD_log_return', 'GBP_USD_prev_log_return', 'GBP_USD_ret_ma5', 'GBP_USD_ret_ma10', 'GBP_USD_ret_ma20', 'GBP_USD_ret_ma60', 'GBP_USD_ret_ma120']

15-paper adds: ['GBP_USD_log', 'GBP_USD_log_sq', 'GBP_USD_log_cube', 'GBP_USD_log_ma5', 'GBP_USD_log_ma10', 'GBP_USD_log_ma20', 'GBP_USD_log_ma60', 'GBP_USD_log_ma220']

15-stationary adds: ['GBP_USD_dev_ma5', 'GBP_USD_dev_ma10', 'GBP_USD_dev_ma20', 'GBP_USD_dev_ma60', 'GBP_USD_dev_ma220', 'GBP_USD_vol20', 'GBP_USD_vol60', 'GBP_USD_z220']


,GBP_USD_log_return,GBP_USD_prev_log_return,GBP_USD_ret_ma5,GBP_USD_ret_ma10,GBP_USD_ret_ma20,GBP_USD_ret_ma60,GBP_USD_ret_ma120,GBP_USD_log,GBP_USD_log_sq,GBP_USD_log_cube,...,GBP_USD_dev_ma5,GBP_USD_dev_ma10,GBP_USD_dev_ma20,GBP_USD_dev_ma60,GBP_USD_dev_ma220,GBP_USD_vol20,GBP_USD_vol60,GBP_USD_z220,GBP_USD_target_return,GBP_USD_target_direction
date,,,,,,,,,,,,,,,,,,,,,
2005-07-29,0.000519,0.007173,0.002182,0.000291,-0.000225,-0.001233,-0.000509,0.563543,0.317581,0.178970,...,0.004941,0.006042,0.003344,-0.024968,-0.056089,0.005944,0.005087,-1.926774,0.005360,1
2005-08-01,0.005360,0.000519,0.002214,0.000829,0.000223,-0.001089,-0.000566,0.568903,0.323650,0.184126,...,0.008088,0.010573,0.008481,-0.018519,-0.050620,0.006013,0.005150,-1.730230,0.002841,1
2005-08-02,0.002841,0.005360,0.003798,0.001818,0.000440,-0.001019,-0.000570,0.571744,0.326891,0.186898,...,0.007131,0.011596,0.010881,-0.014659,-0.047668,0.006026,0.005175,-1.621882,0.003440,1
2005-08-03,0.003440,0.002841,0.003867,0.002105,0.000729,-0.000867,-0.000486,0.575184,0.330837,0.190292,...,0.006704,0.012931,0.013592,-0.010352,-0.044168,0.006025,0.005170,-1.498906,0.001761,1
2005-08-04,0.001761,0.003440,0.002784,0.001589,0.001046,-0.000769,-0.000515,0.576944,0.332865,0.192045,...,0.005681,0.013103,0.014307,-0.007823,-0.042327,0.005896,0.005163,-1.431925,-0.000169,0


In [4]:
splits = {pair: chronological_split(model_datasets[pair]) for pair in PAIRS}
summary = []
for pair in PAIRS:
    for split_name, part in splits[pair].items():
        summary.append({
            "Currency Pair": pair,
            "Split": split_name,
            "Rows": len(part),
            "Start": part.index.min(),
            "End": part.index.max()
        })
display(pd.DataFrame(summary))

,Currency Pair,Split,Rows,Start,End
0,GBP_USD,train,4669,2005-07-29,2023-06-22
1,GBP_USD,val,584,2023-06-23,2025-03-05
2,GBP_USD,test,584,2025-03-06,2026-10-10
3,USD_EUR,train,4669,2005-07-29,2023-06-22
4,USD_EUR,val,584,2023-06-23,2025-03-05
5,USD_EUR,test,584,2025-03-06,2026-10-10
6,USD_JPY,train,4669,2005-07-29,2023-06-22
7,USD_JPY,val,584,2023-06-23,2025-03-05
8,USD_JPY,test,584,2025-03-06,2026-10-10


In [5]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for pair in PAIRS:
    model_datasets[pair].to_csv(PROCESSED_DIR / f"{pair}_model_data.csv")
features_df.to_csv(PROCESSED_DIR / "all_features.csv")
pair_df.to_csv(PROCESSED_DIR / "pair_rates.csv")
print("Saved processed data to:", PROCESSED_DIR)

Saved processed data to: D:\NeuDocuments\SideProject\MiniProject\ML\Foreign-exchange-forecast\data\processed


## Definitions to remember

- **7 features:** current log return, previous log return, and moving averages of returns over 5, 10, 20, 60 and 120 observations.
- **15-paper features:** the original 7 plus log exchange rate, its square and cube, and moving averages of log exchange rate over 5, 10, 20, 60 and 220 observations. These are price *levels*: they drift over the years, so later observations can fall outside the range seen in training.
- **15-stationary features:** the original 7 plus the log price's distance from its 5/10/20/60/220-observation moving averages, 20- and 60-observation return volatility, and a 220-observation z-score of the log price. Same size as the paper's set, but every term is relative to recent history, so its range stays stable.
- **Target return:** log return from observation *t* to *t+1*.
- **Target direction:** 1 if that next return is positive, otherwise 0.
- **Split:** first 80% train, next 10% validation, final 10% test, without shuffling.